# AVMoments-EEG preprocessing

Interactive preprocessing from continuous BrainVision recordings to post-ICA NoReject epochs. Run cells in order and review ICA components before applying exclusions. After inspection, assign the reviewed list to the live ICA_EXCLUDE variable in a new code cell, then continue. Do not rerun initialization just to update this variable, because initialization resets all_epochs.

The default participant is sub-003zgf. Events are read from BrainVision annotations. This notebook does not incorporate event-label corrections stored only in BIDS events.tsv, so it is not the harmonized-event workflow for the first two participants.

Filtering, resampling, epoch timing, ICA fitting, and referencing follow the supplied notebook. Fixed trial alignment, trial rejection, and stimulus-level averaging are separate steps and are not included. See README.md for setup and outputs.


## 1. Configure paths and participant settings


In [ ]:
import mne
import numpy as np
import pandas as pd
from mne.preprocessing import ICA
import gc
import os
import time
import re
import matplotlib.pyplot as plt
from pathlib import Path
from pyprep.find_noisy_channels import NoisyChannels

plt.rcParams['figure.dpi'] = 300

# Timing helpers.
def tic():
    return time.perf_counter()
def toc(t_start):
    print(f'Elapsed time: {time.perf_counter() - t_start:.2f} seconds')


# Run the notebook with this preprocessing directory as the working directory.
# Set DATA_ROOT to the downloaded BIDS_Data directory on your computer.
DATA_ROOT = Path("../../../Data/Zenodo_uploaded/BIDS_Data").resolve()
OUTPUT_ROOT = Path("outputs").resolve()
subid = "sub-003zgf"
SESSION_IDS = list(range(1, 9))

# Session number -> additional bad channels, e.g. {1: ["FC1"]}.
MANUAL_BADS_BY_SESSION = {}
ICA_INSPECT = [0, 1, 2]
# After inspecting components, replace None with your reviewed list.
# Use [] only if you have reviewed the ICA and decided to remove none.
ICA_EXCLUDE = None
OVERWRITE = False

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Set DATA_ROOT to an existing BIDS directory: {DATA_ROOT}")
if not (DATA_ROOT / subid).is_dir():
    raise FileNotFoundError(f"Participant directory not found: {DATA_ROOT / subid}")
if not SESSION_IDS or any(s not in range(1, 9) for s in SESSION_IDS):
    raise ValueError("SESSION_IDS must contain session numbers from 1 to 8.")

# Internal paths used by the processing cells.
data_dir = str(DATA_ROOT) + os.sep
base_dir = str(OUTPUT_ROOT) + os.sep
subject = subid
epoched_dir = str(OUTPUT_ROOT / "Epoched_Data" / subid) + os.sep
ica_dir = str(OUTPUT_ROOT / "ICA" / subid) + os.sep
log_path = str(OUTPUT_ROOT / f"{subid}_preprocessing_log.txt")
Path(epoched_dir).mkdir(parents=True, exist_ok=True)
Path(ica_dir).mkdir(parents=True, exist_ok=True)
all_epochs = []


## 2. Preprocess and epoch each session


In [ ]:

for session_id in SESSION_IDS:
    i = session_id - 1
    
    file_path = data_dir+subid+'/ses-'+str(i+1).zfill(2)+'/eeg/'+subid+'_ses-'+str(i+1).zfill(2)+'_task-video_eeg.vhdr'

    # Skip sessions whose recording file is missing.
    if not Path(file_path).exists():
        print(f"Session {str(i+1).zfill(2)} file not found; skipping")
        continue
        
    raw = mne.io.read_raw_brainvision(file_path, preload=True)
    bip_channels = [ch for ch in raw.ch_names if ch.startswith('BIP')]
    raw.drop_channels(bip_channels)

    print(raw.info)
    raw.info['ch_names']
    raw.set_channel_types({'EOG':'eog'})
    montage = mne.channels.read_custom_montage(data_dir+subid+'/ses-'+str(i+1).zfill(2)+'/eeg/'+subid+'_ses-'+str(i+1).zfill(2)+'_space-CapTrak_electrodes.tsv')
    raw.set_montage(montage)
    # Band-pass filter from 0.03 to 50 Hz.
    raw = raw.filter(l_freq=0.03, h_freq=50)
    # Apply a 50 Hz notch filter with a 4 Hz width for line noise.
    raw.notch_filter(freqs=50, notch_widths=4)
    # Downsample to 100 Hz.
    raw.resample(100, npad="auto")
    
    # Initialize the bad-channel detector.
    nd = NoisyChannels(raw)
    nd.find_all_bads()

    # Retrieve all channels flagged by PyPREP.
    print(f"Channels flagged by PyPREP: {nd.get_bads()}")
    prep_bads = nd.get_bads()
    manual_bads = list(MANUAL_BADS_BY_SESSION.get(session_id, []))
    
    raw.info['bads'] = sorted(set(prep_bads + manual_bads))
    interp_bads = raw.info['bads'].copy()

    # Interpolate bad channels here, including M1/M2 if flagged.
    raw.interpolate_bads(reset_bads=True)
 
    # Optionally inspect the unique annotation labels.
    #unique_desc = np.unique(raw.annotations.description)
    #print(unique_desc)

    # Inspect the recording and extract stimulus events.
    raw.plot(duration=5, n_channels=64, clipping=None)
    events, event_id_raw = mne.events_from_annotations(raw, regexp='.*Stimulus/s([1-9]|[1-5][0-9]|6[0-5]|6[7-9]|[7-9][0-9]|1[0-5][0-9]|16[0-4])$') # Exclude trigger 66, which marks a rest request.

    # Map annotation variants to their numeric trigger values consistently across sessions.
    code_map = {}
    for name, old_code in event_id_raw.items():
        m = re.search(r's(\d+)$', name)
        if m is not None:
            code_map[old_code] = int(m.group(1))

    events_fixed = events.copy()
    for old_code, new_code in code_map.items():
        events_fixed[events[:, 2] == old_code, 2] = new_code

    event_id_fixed = {f"s{k}": k for k in sorted(set(code_map.values()))}

    
    # Optional historical adjustment: remove 20 initial practice trials from session 1 when applicable.
    #if i == 0:
        #events = events[20:]
        
    # Create stimulus-locked epochs.
    epochs = mne.Epochs(raw, events_fixed, event_id=event_id_fixed, tmin=-1.5, tmax=5, baseline=(-0.1, 0),event_repeated='drop', preload=True)
    evoked = epochs.average()
    evoked.plot()

    # Store the session number for each epoch.
    epochs.metadata = pd.DataFrame({'session': [i+1] * len(epochs)})
    
    all_epochs.append(epochs)
    
    save_path = epoched_dir+'ses-'+str(i+1).zfill(2)+'_clean-epochs.fif'
    epochs.save(save_path, overwrite=OVERWRITE)

    # Append session details to the processing log.
    log_msg = (
        f"===== Session {str(i+1).zfill(2)} =====\n"
        f"PyPREP bad channels: {prep_bads}\n"
        f"Additional manual bad channels: {manual_bads}\n"
        f"Channels selected for interpolation: {interp_bads}\n"
        f"Bad channels interpolated: yes, reset_bads=True\n"
        f"Number of events: {len(events_fixed)}\n"
        f"Epoch time window: -1.5 ~ 5 s\n"
        f"Baseline: (-0.1, 0)\n"
        f"{'='*30}\n"
    )
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(log_msg)
    print(log_msg)

    # Save an example EEG epoch plot.
    fig_raw = epochs[100:101].plot(n_epochs=1, n_channels=64, show=False)
    fig_raw_path = epoched_dir + 'ses-' + str(i+1).zfill(2) + '_raw.png'
    fig_raw.savefig(fig_raw_path, dpi=150)
    plt.close('all')

    # Save the session-averaged evoked-response plot.
    fig_evoked = evoked.plot(show=False)
    fig_evoked_path = epoched_dir + 'ses-' + str(i+1).zfill(2) + '_evoked.png'
    fig_evoked.savefig(fig_evoked_path, dpi=150)
    plt.close('all')
    
    print("Session "+str(i+1).zfill(2)+" figures saved")

## 3. Check session epochs


In [ ]:
t = tic()
epochs_list = []
for session_id in SESSION_IDS:
    i = session_id - 1
    path = epoched_dir + f'ses-{str(i+1).zfill(2)}_clean-epochs.fif'
    if not os.path.exists(path):
        print(f"ses-{str(i+1).zfill(2)} file not found; skipping")
        continue
    epochs = mne.read_epochs(path, preload=True)
    epochs_list.append(epochs)
    print(f'===== ses-{str(i+1).zfill(2)} =====')
    print(epochs)
toc(t)

## 4. Combine sessions


In [ ]:
if not epochs_list:
    raise ValueError("No session epochs were loaded. Check paths and SESSION_IDS.")

# Collect event labels and assign consistent IDs across sessions.
unified_event_id = {}
for epochs in epochs_list:
    for key in epochs.event_id:
        if key not in unified_event_id:
            unified_event_id[key] = len(unified_event_id) + 1

print("Unified event_id:", unified_event_id)

# Remap event IDs in each session using the original event array.
fixed_epochs_list = []
for epochs in epochs_list:
    epochs = epochs.copy()
    new_events = epochs.events.copy()
    for key, old_id in epochs.event_id.items():
        new_id = unified_event_id[key]
        new_events[epochs.events[:, 2] == old_id, 2] = new_id
    epochs = mne.EpochsArray(epochs.get_data(),epochs.info,events=new_events,tmin=epochs.tmin,event_id={k: unified_event_id[k] for k in epochs.event_id},baseline=epochs.baseline,metadata=epochs.metadata)
    fixed_epochs_list.append(epochs)

# Concatenate the session epochs.
all_epochs = mne.concatenate_epochs(fixed_epochs_list)
print('===== Concatenation complete =====')
print(all_epochs)

## 5. Inspect electrode positions


In [ ]:
# Inspect electrode positions using the montage assigned during preprocessing.
all_epochs.plot_sensors(show_names=True)

## 6. Fit ICA


In [ ]:
# Use a 0-3 s copy for QC and ICA; retain the full -1.5 to 5 s epochs in all_epochs.
epochs_qc = all_epochs.copy().crop(tmin=0, tmax=3)

t = tic()
ica = ICA(n_components=30, max_iter='auto', random_state=42)
epochs_for_ica = epochs_qc.copy().filter(l_freq=1, h_freq=None) 
ica.fit(epochs_for_ica)
toc(t)

## 7. Inspect ICA components


In [ ]:
# Identify EOG-related components to assist manual review.
eog_indices, eog_scores = ica.find_bads_eog(epochs_for_ica)
print(f"Components with high EOG correlation: {eog_indices}")

# Plot the ICA component maps.
ica.plot_components()

## 8. Inspect selected components


In [ ]:
# sub01: drop IC   0, 1
# sub02: drop IC   0, 1, 2
# sub03: drop IC   0, 1, 3
# sub04: drop IC   0, 1
# sub05: drop IC   0, 1, 2
# sub06: drop IC   0, 3
# sub07: drop IC   0, 1, 2
# sub08: drop IC   0, 1, 2, 3
# sub09: drop IC   
# sub10: drop IC   0, 2
# sub11: drop IC   0, 1
# sub12: drop IC   0, 3
# sub13: drop IC   0, 1, 2, 3

ica.plot_properties(epochs_for_ica, picks=ICA_INSPECT)

## 9. Apply the reviewed ICA exclusions


In [ ]:
if ICA_EXCLUDE is None:
    raise ValueError("Review ICA components and set ICA_EXCLUDE before applying ICA.")
if not isinstance(ICA_EXCLUDE, list) or any(
    not isinstance(k, int) or isinstance(k, bool) or k < 0 or k >= ica.n_components_
    for k in ICA_EXCLUDE
):
    raise ValueError("ICA_EXCLUDE must be a list of valid zero-based component indices.")
ica.exclude = ICA_EXCLUDE.copy()
ica.apply(all_epochs)

del epochs_for_ica
gc.collect()

## 10. Reference to the mastoids


In [ ]:
all_epochs.set_eeg_reference(ref_channels=['M1', 'M2'])

## 11. Plot post-ICA epochs and ERPs


In [ ]:
# Plot one post-ICA epoch from -0.5 to 3 s without trial rejection.
fig1 = all_epochs[100:101].copy().crop(tmin=-0.5, tmax=3).plot(n_epochs=1, n_channels=63)
fig1.savefig(ica_dir + f'{subject}_epochs_AfterICA_NoReject_-0.5_3s.png', dpi=300)

# Plot the mean ERP from -0.5 to 3 s without trial rejection.
evoked = all_epochs.average()
fig3 = evoked.copy().crop(tmin=-0.5, tmax=3).plot()
fig3.savefig(ica_dir + f'{subject}_averaged_AfterICA_NoReject_-0.5_3s.png',dpi=300)

## 12. Save post-ICA data


In [ ]:
# Save the full post-ICA epochs in FIF format without trial rejection.
final_fif_path = ica_dir + f'{subject}_AfterICA_NoReject-epochs.fif'
all_epochs.save(final_fif_path, overwrite=OVERWRITE)

# Save the full post-ICA data array in NumPy format without trial rejection.
epochs_array = all_epochs.get_data()
np.save(ica_dir + f'{subject}_epochs_AfterICA_NoReject.npy', epochs_array)

print(f"Saved FIF: {final_fif_path}")
print(f"Saved NumPy array: {ica_dir + f'{subject}_epochs_AfterICA_NoReject.npy'}")
print(f"epochs_array shape: {epochs_array.shape}")

## 13. Save the processing summary


In [ ]:
log_msg = (
    f"===== {subject} Processing after concatenation (before trial rejection) =====\n"
    f"Full epoch time window: {all_epochs.tmin} ~ {all_epochs.tmax} s\n"
    f"Baseline: {all_epochs.baseline}\n"
    f"ICA time window: 0 ~ 3 s\n"
    f"ICA fitting data: 0-3 s copy with 1 Hz high-pass filtering\n"
    f"Excluded ICA components: {ica.exclude}\n"
    f"Reference: M1/M2\n"
    f"Number of epochs before trial rejection: {len(all_epochs)}\n"
)
with open(log_path, 'a', encoding='utf-8') as f:
    f.write(log_msg + "\n")
print(log_msg)